In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold

from xgboost import XGBRegressor
from scipy.stats import randint

In [2]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [3]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [4]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [5]:
df.shape

(276, 18)

In [6]:
X_train, X_val, y_train, y_val = train_test_split(df, labels, random_state=42, test_size=0.20)

In [7]:
len(X_train)

220

# xgboost


In [8]:
xgb_model = XGBRegressor(random_state=42)

param_dist = {
    'n_estimators': randint(50, 500),  
    'learning_rate': np.linspace(0.01, 0.3, 30),  
    'max_depth': randint(3, 15),  
    'min_child_weight': randint(1, 10), 
    'subsample': np.linspace(0.6, 1.0, 5), 
    'colsample_bytree': np.linspace(0.6, 1.0, 5),  
    'gamma': np.linspace(0, 1, 10) 
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=100,  
     scoring="r2",  
    cv=10,  
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best Hyperparameters: {'colsample_bytree': 0.8, 'gamma': 0.8888888888888888, 'learning_rate': 0.03, 'max_depth': 11, 'min_child_weight': 2, 'n_estimators': 433, 'subsample': 0.7}
Best R2: 0.9674317538738251


In [ ]:
best = random_search.best_params_

fixed_params = {
    "subsample": best["subsample"],
    "colsample_bytree": best["colsample_bytree"],
    "gamma": best["gamma"],
}


grid = {
    "n_estimators": [
        best["n_estimators"] - 60,
        best["n_estimators"] - 30,
        best["n_estimators"],
        best["n_estimators"] + 30,
        best["n_estimators"] + 60,
    ],

    "learning_rate": [
        best["learning_rate"] * 0.7,
        best["learning_rate"] * 0.85,
        best["learning_rate"],
        best["learning_rate"] * 1.15,
        best["learning_rate"] * 1.3,
    ],

    "max_depth": [
        max(2, best["max_depth"] - 2),
        max(2, best["max_depth"] - 1),
        best["max_depth"],
        best["max_depth"] + 1,
    ],

    "min_child_weight": [
        max(1, best["min_child_weight"] - 2),
        max(1, best["min_child_weight"] - 1),
        best["min_child_weight"],
        best["min_child_weight"] + 1,
    ],
}



xgb_local = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    **fixed_params
)

cv = KFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_local,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,         
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)


Fitting 10 folds for each of 400 candidates, totalling 4000 fits
